---

### 🔹 Named Entity Recognition (NER)

**NER** is the process of identifying and classifying specific entities in text into predefined categories such as:

* **Persons** (e.g., *"Barack Obama"*)
* **Organizations** (e.g., *"UNICEF"*)
* **Locations** (e.g., *"New York"*)
* **Dates, Times, Quantities, etc.**

NER tags chunks of text (usually one or more words) with labels like:

* `B-PER` (Beginning of a person name)
* `I-PER` (Inside a person name)
* `O` (Outside any named entity)

#### ✅ Why do we need NER?

* To extract **important information** from unstructured text
* Used in **information retrieval**, **question answering**, **chatbots**, **search engines**, **bioinformatics**, and more

---

### 🔹 Part-of-Speech (POS) Tagging

**POS tagging** assigns a **grammatical category** to each word in a sentence, such as:

* Noun (NN)
* Verb (VB)
* Adjective (JJ)
* Preposition (IN)
* Pronoun (PRP)

#### ✅ Why do we need POS tagging?

* It helps **understand sentence structure**
* Enables **syntactic parsing** and **grammar checking**
* Useful for **machine translation**, **text-to-speech**, **information extraction**, and **sentiment analysis**

---

### 🔍 In Summary:

| Task    | What it does                        | Why it’s useful                                     |
| ------- | ----------------------------------- | --------------------------------------------------- |
| **NER** | Detects and labels named entities   | For extracting real-world entities from text        |
| **POS** | Labels each word's grammatical role | For understanding and processing sentence structure |

Both are **core components of Natural Language Processing (NLP)** and serve as foundational steps in making machines understand human language.


# 1. (LOAD DATASET OR CONVERT IT IN THE DICTIONARY FORMAT)

In [90]:
from datasets import load_dataset

# Load the CoNLL-03 NER dataset with the trust_remote_code flag
dataset = load_dataset("conll2003")

# Check the data
print(dataset["train"][0])


{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}


In [91]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [92]:
from transformers import AutoTokenizer


tokenizer=AutoTokenizer.from_pretrained("bert-base-cased")

C:\Users\Lenovo\AppData\Roaming\Python\Python311\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


# 2. TOKENIZE AND FIX SIZE MISMATCH

## Toknization issue

when tokenizing the wordlists, 2 things can happen
- 1. The word remains same
- 2. The word is splitted into subwords

When words are splitted into subwords, no additional labels are created for them, as we know
<center>num of tokens = num of ner tags</center>
but we have <center>num of tokens > num of ner tags </center/>
to fix that we need to do label alignment.

Let me explain **clearly and step-by-step** what’s going on and how to connect everything.

---

## ✅ What You’re Seeing

```python
dataset["train"][0]
```

returns something like:

```python
{
  'id': '0',
  'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'],
  'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
  'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
  'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]
}
```
---

## 📌 Important Points:

1. **`tokens`** = Original words
2. **`ner_tags`** = Integer-encoded BIO tags (for Named Entities)
3. **`ner_tags` are per word** — so you **must align them** to BERT's **subword tokens** using `word_ids`.

---

## ✅ How to Map NER Labels

You can map the tag integers to actual tags using the dataset's `.features`:

```python
label_names = dataset["train"].features["ner_tags"].feature.names
print(label_names)
```

Output:

```python
['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
```

So in your example:

```python
ner_tags = [3, 0, 7, 0, 0, 0, 7, 0, 0]
```

Maps to:

```python
['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']
```

---


In [106]:
tokenizer.tokenize(dataset["train"][0]["tokens"],is_split_into_words=True), dataset["train"][0]["ner_tags"]

(['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'la',
  '##mb',
  '.'],
 [3, 0, 7, 0, 0, 0, 7, 0, 0])

 Here, <center><b> the number of tokens != ner tags</b> </center>


In [94]:
sample=tokenizer(dataset["train"][0]["tokens"], is_split_into_words=True)
sample

{'input_ids': [101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

## [SEP] and [CLS] issue

Bert tokenizer adds <b>CLS</b> and <b>SEP</b> ids before and after of each sentence. This increases input ids and makes
<center> input_ids > target_ner_labels</center>

<img src="word_id_positioning.png" alt="alt text" width="1000" height="500">

In [108]:
sample.tokens(),sample.word_ids()

(['[CLS]',
  'EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'la',
  '##mb',
  '.',
  '[SEP]'],
 [None, 0, 1, 2, 3, 4, 5, 6, 7, 7, 8, None])

Here <b>CLS</b> and <b>SEP</b> are given None values, again since lamb is devided into <b> <i>la, ##mb</i> </b> thus, they are provided with the same ids

## IOB format issue
<img src="iob_format.png" alt="alt text" width="1000" height="500">

In the IOB (Inside, Outside, Beginning) format, the handling of multi-token words (i.e., when a word is split into multiple subwords during tokenization) follows a specific approach to maintain the entity boundaries:

1. **Beginning of Entity (B)**: The first subword gets the "B-" prefix, which indicates the beginning of an entity.
2. **Inside Entity (I)**: The subsequent subwords within the entity receive the "I-" prefix, indicating they are part of the same entity.
3. **Outside Entity (O)**: Any token outside the entity gets the "O" label.

For example, if the name "Steve" is split into tokens like "Ste" and "##ve," then the labeling would be:

* "Ste" → `B-PERSON` (Beginning of a person)
* "##ve" → `I-PERSON` (Inside the same person entity)

### If there are more than two subwords:

Let’s say a name is split into three or more subwords, for example, "Maxwell" being split into \["Max", "##well", "##son"].

Here’s how the labeling would work:

* "Max" → `B-PERSON` (Beginning of the entity "Maxwell")
* "##well" → `I-PERSON` (Inside the entity "Maxwell")
* "##son" → `I-PERSON` (Still inside the same entity "Maxwell")

### Key Points:

* The first subword gets a "B-" tag (Beginning of the entity).
* Any subsequent subwords for the same word are tagged with "I-" (Inside the entity).
* This continues until the entity ends (the next word or token will be labeled with "O" if it's not part of an entity).

If there are more than 3 subwords, the same approach applies; each subsequent token gets the "I-" tag until all subwords are processed.



In [96]:
possible_ner_tags = dataset["train"].features["ner_tags"].feature
possible_ner_tags

ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC'], id=None)

## NER LABEL alignment code

In [ ]:
def align_ner_target(word_ids, ner_tags):
    aligned_labels = []
    prev_word_id = None
    
    for word_id in word_ids:
        if word_id is None:
            aligned_labels.append(-100)  # Use -100 to ignore this token in loss computation (this is None in the original list)

        elif word_id != prev_word_id: # if the word is not none and didnot appear before
            aligned_labels.append(ner_tags[word_id])
            prev_word_id = word_id # update the current word id
            
        else:  # repeated subword
            # Subword token (repeated word_id)
            label = ner_tags[word_id]
            if label != 0 and label % 2 == 1: # odd positions are B-PER, B - ORG ,B-LOC, B-MISC,so deviding by their index will give us odd number 
                aligned_labels.append(label + 1)  # Convert (B-XXX → I-XXX)
            else:
                aligned_labels.append(label)  # O or already (I-XXX)
                
    return aligned_labels


---

### 🔹 Short Note on `align_ner_target()` Function

The `align_ner_target()` function aligns NER labels to tokenized inputs produced by BERT. Since BERT splits some words into subwords, this function ensures each subword gets the correct label:

* The **first token** of a word gets its original NER tag.
* **Subword tokens** (repeated word\_ids) convert `B-XXX` tags to `I-XXX` by adding 1 to the tag ID.
* Special tokens like `[CLS]` and `[SEP]` are assigned `-100` to be ignored during training.

This ensures accurate label alignment for token-level NER training using BIO tagging format.

---

## ✅ Your Given Data:

```python
ner_tags = [3, 0, 7, 0, 0, 0, 7, 0, 0]
word_ids = [None, 0, 1, 2, 3, 4, 5, 6, 7, 7, 8, None]
same_tag_converter = {1: 2, 3: 4, 5: 6, 7: 8}
```

---

### 🔢 Step-by-Step Alignment:

| idx | word\_id | current\_word\_id | Action                                         | NER Tags Used       | Final NER tags |
| --- | -------- | ----------------- | ---------------------------------------------- | ---------------- | ----------- |
| 0   | None     | None              | Padding → -100                                 | -                | -100        |
| 1   | 0        | None              | First time → `ner_tags[0]`                     | 3                | 3           |
| 2   | 1        | 0                 | First time → `ner_tags[1]`                     | 0                | 0           |
| 3   | 2        | 1                 | First time → `ner_tags[2]`                     | 7                | 7           |
| 4   | 3        | 2                 | First time → `ner_tags[3]`                     | 0                | 0           |
| 5   | 4        | 3                 | First time → `ner_tags[4]`                     | 0                | 0           |
| 6   | 5        | 4                 | First time → `ner_tags[5]`                     | 0                | 0           |
| 7   | 6        | 5                 | First time → `ner_tags[6]`                     | 7                | 7           |
| 8   | 7        | 6                 | First time → `ner_tags[7]`                     | 0                | 0           |
| 9   | 7        | 7                 | Repeated → `7 in converter`, use `ner_tags[8]` | 0 (from index 8) | 0           |
| 10  | 8        | 7                 | First time → `ner_tags[8]`                     | 0                | 0           |
| 11  | None     | 8                 | Padding → -100                                 | -                | -100        |

---

### ✅ Final Output:

```python
aligned_labels = [-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, 0, -100]
```


### 🔍 Why is this important?

At index 9 (second `word_id = 7`), you correctly used:

```python
new_id = same_tag_converter[7] = 8
ner_tags[8] = 0
```
---

In [137]:
def tokenizer_fun(data):
    tokenized_input = tokenizer(data["tokens"], is_split_into_words=True, truncation=True) #tokenize the input

    old_ner_labels = data["ner_tags"]   #get the old labels
    aligned_ner_labels = []
    for i,ner_labels in enumerate(old_ner_labels): #iterate through the old labels list
        word_ids=tokenized_input.word_ids(i)
        aligned_ner_labels.append(align_ner_target(word_ids, ner_labels)) #align the labels with the tokenized input word ids
    tokenized_input["labels"] = aligned_ner_labels #add the aligned labels to the tokenized input
    return tokenized_input

In [138]:
sample.word_ids() # tokenized word ids

[None, 0, 1, 2, 3, 4, 5, 6, 7, 7, 8, None]

In [139]:
dataset["train"]["ner_tags"][0] # old ner tags

[3, 0, 7, 0, 0, 0, 7, 0, 0]

In [140]:
aligned_labels = align_ner_target(sample.word_ids(), dataset["train"]["ner_tags"][0])
aligned_labels

[-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, 0, -100]

In [143]:
tokenizer.tokenize(dataset["train"][2]["tokens"],is_split_into_words=True), dataset["train"][2]["tokens"],dataset["train"][2]["ner_tags"]


(['BR', '##US', '##SE', '##LS', '1996', '-', '08', '-', '22'],
 ['BRUSSELS', '1996-08-22'],
 [5, 0])

In [ ]:
tokenized_df= dataset.map(
    tokenizer_fun,
    batched=True,
    remove_columns=dataset["train"].column_names, # remove the original columns to keep only the tokenized ones
)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [146]:
tokenized_df["train"].features

{'input_ids': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None),
 'token_type_ids': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None),
 'attention_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None),
 'labels': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None)}

In [148]:
tokenized_df

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})

# 3. DATA COLLATOR SETUP FOR BATCHED PREPROCESS

<img src="data_collator.png" alt="alt text" width="1000" height="500">

In [149]:
from transformers import DataCollatorForTokenClassification

data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer)

In [154]:
%pip install seqeval

### old method, load_metric has been depricated

In [ ]:
# from datasets import load_metric
# metric = load_metric("seqeval")

ImportError: cannot import name 'load_metric' from 'datasets' (c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\datasets\__init__.py)

In [156]:
%pip install evaluate


Note: you may need to restart the kernel to use updated packages.


In [157]:
import evaluate

metric = evaluate.load("seqeval")


## Data Collator format (list of dictionaries)

In [163]:
[tokenized_df["train"][0] ]

[{'input_ids': [101,
   7270,
   22961,
   1528,
   1840,
   1106,
   21423,
   1418,
   2495,
   12913,
   119,
   102],
  'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
  'labels': [-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, 0, -100]}]

## But when we loop through, we get dicionary of lists (opposite of collator format)

In [165]:
tokenized_df["train"][:3]

{'input_ids': [[101,
   7270,
   22961,
   1528,
   1840,
   1106,
   21423,
   1418,
   2495,
   12913,
   119,
   102],
  [101, 1943, 14428, 102],
  [101, 26660, 13329, 12649, 15928, 1820, 118, 4775, 118, 1659, 102]],
 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  [0, 0, 0, 0],
  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]],
 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
  [1, 1, 1, 1],
  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
 'labels': [[-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, 0, -100],
  [-100, 1, 2, -100],
  [-100, 5, 6, 6, 6, 0, 0, 0, 0, 0, -100]]}

## Required way

In [166]:
[tokenized_df["train"][i] for i in range(3)]

[{'input_ids': [101,
   7270,
   22961,
   1528,
   1840,
   1106,
   21423,
   1418,
   2495,
   12913,
   119,
   102],
  'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
  'labels': [-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, 0, -100]},
 {'input_ids': [101, 1943, 14428, 102],
  'token_type_ids': [0, 0, 0, 0],
  'attention_mask': [1, 1, 1, 1],
  'labels': [-100, 1, 2, -100]},
 {'input_ids': [101,
   26660,
   13329,
   12649,
   15928,
   1820,
   118,
   4775,
   118,
   1659,
   102],
  'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
  'labels': [-100, 5, 6, 6, 6, 0, 0, 0, 0, 0, -100]}]

In [169]:
test_collated_data = data_collator([tokenized_df["train"][i] for i in range(3)])
test_collated_data["labels"] # Automatic padding and adding -100 for the extra padded tokens, they will be 0 while computing loss  


tensor([[-100,    3,    0,    7,    0,    0,    0,    7,    0,    0,    0, -100],
        [-100,    1,    2, -100, -100, -100, -100, -100, -100, -100, -100, -100],
        [-100,    5,    6,    6,    6,    0,    0,    0,    0,    0, -100, -100]])

**`seqeval`** is a popular Python library used to **evaluate sequence labeling tasks**, especially for **Named Entity Recognition (NER)**. It calculates metrics like **precision**, **recall**, and **F1-score** based on **entity-level** evaluation, not just individual token-level accuracy.

---

## 🔍 What is `seqeval` used for?

Primarily for **NER model evaluation**, where the goal is to measure how accurately the model identifies complete entities like:

* "B-PER I-PER" → *a person name*
* "B-LOC I-LOC" → *a location*

Unlike token-wise accuracy, `seqeval` checks if **entire entities** were correctly predicted — both in span and label.

---

## ✅ Example: How it works

### 🎯 Inputs:

You pass **two lists**:

* `predictions` = Model output
* `references` = Ground truth

Each list contains sequences of NER tags.

```python
import evaluate

metric = evaluate.load("seqeval")

predictions = [['B-PER', 'I-PER', 'O', 'B-LOC']]
references   = [['B-PER', 'I-PER', 'O', 'B-LOC']]

results = metric.compute(predictions=predictions, references=references)

print(results)
```

### 📤 Output:

```python
{
  'overall_precision': 1.0,
  'overall_recall': 1.0,
  'overall_f1': 1.0,
  'overall_accuracy': 1.0,
  'PER': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 1},
  'LOC': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 1}
}
```

---

## 🧠 How does it evaluate?

It **extracts named entities** using tag formats like:

* `BIO` (Beginning, Inside, Outside)
* `BILOU` (Beginning, Inside, Last, Outside, Unit)

Then it compares:

* **Predicted spans** (start to end of entities)
* **Entity type** (e.g., PER, LOC)

✔️ A correct prediction must match **both** entity boundaries and label.

---

## 🧾 Supported Formats

* `BIO`, `IOB2`, `BILOU`
* Tags must be strings like `'B-ORG'`, `'I-PER'`, `'O'`

---

## ✅ Summary

| Feature          | Description                                                 |
| ---------------- | ----------------------------------------------------------- |
| Purpose          | Evaluation of NER and sequence labeling tasks               |
| Works on         | Entity-level, not just token-level                          |
| Input            | List of tag sequences (predictions and references)          |
| Metrics provided | Precision, Recall, F1, Accuracy (overall + per-entity type) |

---

## sequeval requires 2d matrix comparison of IOB OUTPUTS

In [173]:
metric.compute(
    predictions=[["B-PER","O","B-ORG"]],
    references=[["B-PER","I-PER","B-ORG"]],
)

{'ORG': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 1},
 'PER': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 1},
 'overall_precision': 0.5,
 'overall_recall': 0.5,
 'overall_f1': 0.5,
 'overall_accuracy': 0.6666666666666666}

In [188]:
id2label = {
    0: "O",        # Outside any named entity
    1: "B-PER",    # Beginning of a Person
    2: "I-PER",    # Inside a Person
    3: "B-ORG",    # Beginning of an Organization
    4: "I-ORG",    # Inside an Organization
    5: "B-LOC",    # Beginning of a Location
    6: "I-LOC",    # Inside a Location
    7: "B-MISC",   # Beginning of Miscellaneous (optional in some datasets)
    8: "I-MISC"    # Inside of Miscellaneous
}

label2id = {v: k for k, v in id2label.items()}

In [180]:
seqeval = evaluate.load("seqeval")

# 4. METRICS SETUP

In [189]:
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)  # Convert logits to label ids

    true_predictions = []
    true_labels = []

    for pred, label in zip(predictions, labels):
        temp_preds = []
        temp_labels = []
        for p_i, l_i in zip(pred, label):
            if l_i != -100:
                temp_preds.append(id2label[p_i])
                temp_labels.append(id2label[l_i])
        true_predictions.append(temp_preds)
        true_labels.append(temp_labels)
        
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }


# 4. TRAIN

In [190]:
from transformers import AutoModelForTokenClassification


model=AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    id2label=id2label,
    label2id=label2id,)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import TrainingArguments

training_args= TrainingArguments(
    output_dir="ner_tags",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,   
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_df["train"],
    eval_dataset=tokenized_df["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    
)

In [196]:
trainer.train()

  0%|          | 0/878 [00:00<?, ?it/s]

{'loss': 0.1878, 'grad_norm': 1.0563019514083862, 'learning_rate': 8.610478359908885e-06, 'epoch': 0.57}


c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  0%|          | 0/204 [00:00<?, ?it/s]

{'eval_loss': 0.06735745072364807, 'eval_precision': 0.893848401233566, 'eval_recall': 0.9267923258162235, 'eval_f1': 0.9100223085185492, 'eval_accuracy': 0.9808677223759346, 'eval_runtime': 137.7285, 'eval_samples_per_second': 23.597, 'eval_steps_per_second': 1.481, 'epoch': 1.0}
{'train_runtime': 2582.7396, 'train_samples_per_second': 5.436, 'train_steps_per_second': 0.34, 'train_loss': 0.14064029789188184, 'epoch': 1.0}


TrainOutput(global_step=878, training_loss=0.14064029789188184, metrics={'train_runtime': 2582.7396, 'train_samples_per_second': 5.436, 'train_steps_per_second': 0.34, 'total_flos': 351240792638148.0, 'train_loss': 0.14064029789188184, 'epoch': 1.0})

In [198]:
trainer.save_model("ner_tags")

In [199]:
from transformers import pipeline

ner=pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",  # Use simple aggregation to combine subword tokens
)

In [201]:
s=ner("Hugging Face is based in New York City and Paris.")
s

[{'entity_group': 'ORG',
  'score': 0.76966286,
  'word': 'Hugging Face',
  'start': 0,
  'end': 12},
 {'entity_group': 'LOC',
  'score': 0.9886517,
  'word': 'New York City',
  'start': 25,
  'end': 38},
 {'entity_group': 'LOC',
  'score': 0.99608624,
  'word': 'Paris',
  'start': 43,
  'end': 48}]